# Task 1 · Exploratory Data Analysis on Retail Sales Data
**Objective:** explore a retail sales dataset to uncover patterns, customer behaviour trends and actionable business insights.

**Business questions**
1. Which products and categories drive the most sales and profit?
2. What is the trend of sales over time (monthly / quarterly / yearly)?
3. Which stores or regions perform best?
4. How do customer demographics (age, gender, city) influence sales?
5. What is the average order value and customer lifetime value?
6. How do discounts affect revenue?

This notebook covers **Part 1: data loading, cleaning and feature engineering**. Later parts answer each question in turn.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

DATA = Path("../data/retail_sales_dataset.xlsx")
OUT = Path("../outputs")
(OUT / "figures").mkdir(parents=True, exist_ok=True)

## 2. Load the data
The workbook is a small relational database spread over four sheets: **Customers**, **Products**, **Stores** and **Transactions** (one row per sale line). Transactions links to the other three through `CustomerID`, `ProductID` and `StoreID`.

In [2]:
sheets = pd.read_excel(DATA, sheet_name=None)
customers, products = sheets["Customers"], sheets["Products"]
stores, tx = sheets["Stores"], sheets["Transactions"]

for name, df in sheets.items():
    print(f"{name:<13} rows={df.shape[0]:>5}  cols={df.shape[1]}")

Customers     rows=  200  cols=7
Products      rows=   50  cols=6
Stores        rows=    5  cols=4
Transactions  rows= 5000  cols=8


## 3. Data quality checks

In [3]:
checks = {}
for name, df in sheets.items():
    checks[name] = {
        "rows": len(df),
        "missing values": int(df.isna().sum().sum()),
        "duplicate rows": int(df.duplicated().sum()),
    }
pd.DataFrame(checks).T

,rows,missing values,duplicate rows
Customers,200,0,0
Products,50,0,0
Stores,5,0,0
Transactions,5000,0,0


In [4]:
# Primary keys must be unique
for df, key in [(customers, "CustomerID"), (products, "ProductID"),
                (stores, "StoreID"), (tx, "TransactionID")]:
    assert df[key].is_unique, f"{key} is not unique"

# Foreign keys in Transactions must exist in the lookup tables (no orphans)
print("Orphan CustomerIDs:", (~tx.CustomerID.isin(customers.CustomerID)).sum())
print("Orphan ProductIDs :", (~tx.ProductID.isin(products.ProductID)).sum())
print("Orphan StoreIDs   :", (~tx.StoreID.isin(stores.StoreID)).sum())

# Business-rule checks
print("Products priced below cost:", (products.CostPrice > products.UnitPrice).sum())
print("Quantity range:", tx.Quantity.min(), "to", tx.Quantity.max())
print("Discount values:", sorted(tx.Discount.unique()))
print("Date range:", tx.Date.min().date(), "to", tx.Date.max().date())

Orphan CustomerIDs: 0
Orphan ProductIDs : 0
Orphan StoreIDs   : 0
Products priced below cost: 0
Quantity range: 1 to 5
Discount values: [np.float64(0.0), np.float64(0.05), np.float64(0.1), np.float64(0.15)]
Date range: 2023-09-10 to 2025-09-09


**Findings so far:** no missing values, no duplicates, no orphan keys, and prices and discounts are all valid. The data is structurally clean.

One more check: sales that happen *before* the customer's recorded `JoinDate`.

In [5]:
merged_check = tx.merge(customers[["CustomerID", "JoinDate"]], on="CustomerID")
before_join = merged_check[merged_check.Date < merged_check.JoinDate]
print(f"Transactions dated before the customer's JoinDate: {len(before_join)} "
      f"({len(before_join)/len(tx):.1%} of all transactions)")
print("Customers affected:", before_join.CustomerID.nunique(), "of", customers.CustomerID.nunique())

Transactions dated before the customer's JoinDate: 996 (19.9% of all transactions)
Customers affected: 75 of 200


**Decision:** about one in five transactions predate the customer's join date. In a real business this would suggest JoinDate is the date of loyalty-programme sign-up rather than first purchase. Because the sales themselves are valid, I **keep these rows** and add a flag column `BeforeJoinDate`. I therefore avoid using JoinDate for anything beyond a note, and customer lifetime value is based on actual transactions instead.

## 4. Merge the tables
A left join keeps every transaction and attaches product, customer and store details.

In [6]:
df = (tx
      .merge(products, on="ProductID", how="left")
      .merge(customers, on="CustomerID", how="left")
      .merge(stores, on="StoreID", how="left", suffixes=("_Customer", "_Store")))
df = df.rename(columns={"City_Customer": "CustomerCity", "City_Store": "StoreCity"})
assert len(df) == len(tx), "Merge changed the number of rows"
df.head()

,TransactionID,Date,CustomerID,ProductID,StoreID,Quantity,Discount,PaymentMethod,ProductName,Category,SubCategory,UnitPrice,CostPrice,FirstName,LastName,Gender,BirthDate,CustomerCity,JoinDate,StoreName,StoreCity,Region
0,T00001,2024-06-18,C160,P014,S003,1,0.10,Bank Transfer,Add Clothing,Fashion,Clothing,"1,342.75",797.94,Meagan,Macdonald,M,1991-09-10,Port Jacob,2021-10-27,MegaMart New Michele,New Michele,West
1,T00002,2023-11-02,C171,P030,S004,3,0.15,Bank Transfer,National Watches,Fashion,Watches,29.24,15.28,Christina,Dominguez,M,1993-03-15,Robertsborough,2025-07-28,MegaMart Brianahaven,Brianahaven,North
2,T00003,2024-03-28,C142,P002,S002,2,0.15,Mobile Money,Audience Television,Electronics,Television,818.76,527.62,Suzanne,Fox,M,2005-07-25,Joelborough,2020-12-27,MegaMart Peckmouth,Peckmouth,East
3,T00004,2024-06-15,C174,P050,S002,5,0.10,Mobile Money,Maybe Footwear,Fashion,Footwear,"1,044.64",775.07,Scott,Howell,M,2005-07-07,South Jennifer,2025-08-28,MegaMart Peckmouth,Peckmouth,East
4,T00005,2024-08-29,C141,P036,S001,3,0.10,Credit Card,New Watches,Fashion,Watches,"1,501.46","1,167.73",Adam,Lucas,F,1978-08-06,Allisonmouth,2024-01-07,MegaMart Jimenezborough,Jimenezborough,South


## 5. Feature engineering
| Feature | Formula |
|---|---|
| `GrossSales` | Quantity × UnitPrice |
| `DiscountAmount` | GrossSales × Discount |
| `Revenue` | GrossSales − DiscountAmount (what the customer actually pays) |
| `TotalCost` | Quantity × CostPrice |
| `Profit` | Revenue − TotalCost |
| `ProfitMargin` | Profit ÷ Revenue |
| `Year`, `Quarter`, `Month`, `YearMonth` | taken from `Date` |
| `Age` | age of the customer on the transaction date |
| `AgeGroup` | binned age |

In [7]:
df["GrossSales"] = df.Quantity * df.UnitPrice
df["DiscountAmount"] = df.GrossSales * df.Discount
df["Revenue"] = df.GrossSales - df.DiscountAmount
df["TotalCost"] = df.Quantity * df.CostPrice
df["Profit"] = df.Revenue - df.TotalCost
df["ProfitMargin"] = df.Profit / df.Revenue

df["Year"] = df.Date.dt.year
df["Quarter"] = df.Date.dt.to_period("Q").astype(str)
df["Month"] = df.Date.dt.month
df["YearMonth"] = df.Date.dt.to_period("M").astype(str)

df["Age"] = ((df.Date - df.BirthDate).dt.days // 365.25).astype(int)
df["AgeGroup"] = pd.cut(df.Age, bins=[0, 24, 34, 44, 54, 64, 120],
                        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"])
df["BeforeJoinDate"] = df.Date < df.JoinDate
df["DiscountBand"] = (df.Discount * 100).round().astype(int).astype(str) + "%"

# Sanity checks on the new columns
assert (df.Revenue >= 0).all()
assert np.isclose(df.Revenue + df.DiscountAmount, df.GrossSales).all()
print("Negative-profit rows:", (df.Profit < 0).sum())
df[["TransactionID", "Quantity", "UnitPrice", "Discount", "Revenue", "Profit", "ProfitMargin", "Age", "AgeGroup"]].head()

Negative-profit rows: 0


,TransactionID,Quantity,UnitPrice,Discount,Revenue,Profit,ProfitMargin,Age,AgeGroup
0,T00001,1,"1,342.75",0.10,"1,208.47",410.53,0.34,32,25-34
1,T00002,3,29.24,0.15,74.56,28.72,0.39,30,25-34
2,T00003,2,818.76,0.15,"1,391.89",336.65,0.24,18,<25
3,T00004,5,"1,044.64",0.10,"4,700.88",825.53,0.18,18,<25
4,T00005,3,"1,501.46",0.10,"4,053.94",550.75,0.14,46,45-54


## 6. Overview of the cleaned data

In [8]:
print("Shape:", df.shape)
print(f"Total revenue : {df.Revenue.sum():>14,.2f}")
print(f"Total profit  : {df.Profit.sum():>14,.2f}")
print(f"Overall margin: {df.Profit.sum()/df.Revenue.sum():>14.1%}")
df[["Quantity", "Discount", "Revenue", "Profit", "ProfitMargin", "Age"]].describe().T

Shape: (5000, 36)
Total revenue :  14,301,903.15
Total profit  :   3,826,314.67
Overall margin:          26.8%


,count,mean,std,min,25%,50%,75%,max
Quantity,"5,000.00",2.99,1.41,1.00,2.00,3.00,4.00,5.00
Discount,"5,000.00",0.08,0.06,0.00,0.05,0.10,0.15,0.15
Revenue,"5,000.00","2,860.38","2,265.96",21.73,"1,043.15","2,156.12","4,451.31","9,763.25"
Profit,"5,000.00",765.26,711.18,2.68,248.91,555.11,"1,066.50","4,372.60"
ProfitMargin,"5,000.00",0.28,0.11,0.06,0.19,0.27,0.37,0.50
Age,"5,000.00",43.41,15.42,16.00,30.00,44.00,58.00,70.00


## 7. Save the cleaned dataset
Saved so the analysis notebooks can start from here.

In [9]:
df.to_csv(OUT / "retail_clean.csv", index=False)
print("Saved", OUT / "retail_clean.csv", df.shape)

Saved ../outputs/retail_clean.csv (5000, 36)
